In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import models, transforms
import cv2
from sklearn.preprocessing import LabelEncoder
from torchvision import datasets
from torch.utils.data import DataLoader, ConcatDataset
import requests
import zipfile

In [ ]:
BACKEND_URL = "http://YOUR_NGROK_URL/api" 
API_KEY = "my_super_secret_key_123"

print(f"Connecting to Local Backend at {BACKEND_URL}...")
live_data_dir = './live_data'
headers = {"x-api-key": API_KEY}

In [ ]:
try:
    response = requests.get(f"{BACKEND_URL}/export-data", headers=headers)
    if response.status_code == 200:
        with open("verified_data.zip", "wb") as f:
            f.write(response.content)
        with zipfile.ZipFile("verified_data.zip", 'r') as zip_ref:
            zip_ref.extractall(live_data_dir)
        print("Successfully downloaded user-verified data from Local Backend!")
    else:
        print("No new verified data available on backend.")
except Exception as e:
    print(f"Could not connect to backend: {e}")

In [ ]:
class PlantDiseaseClassifier(nn.Module):
    def __init__(self, num_classes=38):
        super().__init__()
        self.model = models.efficientnet_b0(weights='DEFAULT')
        in_features = self.model.classifier[1].in_features
        self.model.classifier = nn.Identity()
        self.fc1 = nn.Linear(in_features, 512)
        self.relu1 = nn.ReLU()
        self.drop1 = nn.Dropout(0.4)
        self.fc2 = nn.Linear(512, 128)
        self.relu2 = nn.ReLU()
        self.drop2 = nn.Dropout(0.3)
        self.fc3 = nn.Linear(128, num_classes)
    
    def forward(self, x):
        x = self.model(x)
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.drop1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.drop2(x)
        x = self.fc3(x)
        return x

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
train_path = '/kaggle/input/datasets/vipoooool/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train'
val_path = '/kaggle/input/datasets/vipoooool/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/valid'

if not os.path.exists(train_path):
    print("Kaggle path not found. Downloading dataset via kagglehub...")
    import kagglehub
    dataset_path = kagglehub.dataset_download("vipoooool/new-plant-diseases-dataset")
    train_path = os.path.join(dataset_path, "New Plant Diseases Dataset(Augmented)", "New Plant Diseases Dataset(Augmented)", "train")
    val_path = os.path.join(dataset_path, "New Plant Diseases Dataset(Augmented)", "New Plant Diseases Dataset(Augmented)", "valid")

train_dataset = datasets.ImageFolder(root=train_path, transform=transform)
val_dataset = datasets.ImageFolder(root=val_path, transform=transform)

try:
    if os.path.exists(live_data_dir) and len(os.listdir(live_data_dir)) > 0:
        live_dataset = datasets.ImageFolder(live_data_dir, transform=transform)
        full_train_dataset = ConcatDataset([train_dataset, live_dataset])
        print("Merged Kaggle Data + Backend Live Data!")
    else:
        full_train_dataset = train_dataset
except Exception:
    full_train_dataset = train_dataset
    
    
train_loader = DataLoader(full_train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
class_names = train_dataset.classes


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PlantDiseaseClassifier(num_classes=len(class_names)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 50
patience = 5  
best_val_loss = float('inf')
counter = 0


for epoch in range(epochs):
    model.train()  
    train_loss = 0.0
    train_correct = 0 
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        _, train_predicted = torch.max(outputs, 1)
        train_correct += (train_predicted == labels).sum().item()
        
    